# Water Rights Restored to the Gila River

The impacts of irrigation on vegetation health in the Gila River Valley


## Set up

To get started on this notebook, you’ll need to restore any variables from previous notebooks to your workspace. To save time and memory, make sure to specify which variables you want to load.

In [1]:
%store -r

Unable to restore variable 'ndvi_diff', ignoring (use %store -d to forget!)
The error was: <class 'KeyError'>


You will also need to import any libraries you are using in this
notebook, since they won’t carry over from the previous notebook:

In [2]:
### import libraries
### working with dataframes
import pandas as pd

### working with geodataframes
import geopandas as gpd

### making interactive plots of tabular and vector data
import hvplot.pandas 

### work with json data
import json

### work with pattern matching rules for pathnames 
from glob import glob

### access the data we'll use for the assignment
import earthpy

### interactive plots with rasters 
import hvplot.xarray

### handle geospatial rasters
import rioxarray as rxr

### work with arrays
import xarray as xr

### display interactive plot in notebook
import panel as pn
pn.extension()

print("Libraries imported succesfully")

Libraries imported succesfully


In [3]:
### solution to version mismatch causing coordinate system key error (Online)
import cartopy.crs as ccrs

### save cartopy original version once
if not hasattr(ccrs.PlateCarree, '_original_proj4_params'):
    ccrs.PlateCarree._original_proj4_params = ccrs.PlateCarree.proj4_params

def _proj4_params_with_lon_0(self):
    params = ccrs.PlateCarree._original_proj4_params.fget(self)
    params.setdefault('lon_0', params.get('pm', 0))
    return params

ccrs.PlateCarree.proj4_params = property(_proj4_params_with_lon_0)

## Step 5: Is the mean annual July NDVI different within the Gila River Indian Community after the change in water rights?

You will compute the mean NDVI inside and outside the Reservation boundary. 

### Step 5A: Get the NDVI inside and outside the Gila River Indian Community

First, use the code below to get a `GeoDataFrame` of the area outside the Reservation.

- Check the variable names. Make sure that the code uses your boundary `GeoDataFrame`  
- How could you test if the geometry was modified correctly? Add some code to take a look at the results

In [4]:
### compute the area outside the reservation boundary
out_gdf = (
    gpd.GeoDataFrame(geometry = gilaboundary_gdf.envelope)
    .overlay(gilaboundary_gdf, how = 'difference'))
out_gdf

,geometry
0,"MULTIPOLYGON (((-112.30875 32.96704, -112.3087..."


Next, clip your DataArray to the boundaries for both inside and outside
the reservation. You will need to replace the `GeoDataFrame` name with
your own. Check out the [lesson on clipping data with the `rioxarray`
library on EarthDataScience.org](https://www.earthdatascience.org/courses/use-data-open-source-python/intro-raster-data-python/raster-data-processing/crop-raster-data-with-shapefile-in-python/).

**Note**: It’s important to use `from_disk = True` when clipping large arrays like this. It allows the computer to use less valuable memory resources when clipping. Otherwise, you will probably find that the cell below crashes your kernel.

In [5]:
### clip data to inside the reservation's boundaries
ndvi_in_da = ndvi_da.rio.clip(gilaboundary_gdf.geometry, from_disk=True)

### clip data to outside the reservation's boundaries 
ndvi_out_da = ndvi_da.rio.clip(out_gdf.geometry, from_disk=True)

ndvi_in_da

<xarray.Dataset> Size: 47MB
Dimensions:      (x: 379, y: 202, date: 154)
Coordinates:
  * x            (x) float64 3kB -112.3 -112.3 -112.3 ... -111.5 -111.5 -111.5
  * y            (y) float64 2kB 33.39 33.38 33.38 33.38 ... 32.97 32.97 32.97
  * date         (date) datetime64[us] 1kB 2001-01-14 2001-01-16 ... 2022-01-24
    band         int64 8B 1
    spatial_ref  int64 8B 0
Data variables:
    NDVI         (date, y, x) float32 47MB nan nan nan nan ... nan nan nan nan

### Step 5B: Assess the change in NDVI before and after the change in water rights, inside and outside the boundaries of the Gila River Indian Community

For **both inside and outside** the Gila River Indian Community boundary, calculate the mean annual July NDVI:  

- Group the data by year  
- Take the mean. You always need to tell reducing methods in `xarray` what dimensions you want to reduce. When you want to summarize data across **all** dimensions, you can use the `...` syntax, e.g. `.mean(...)` as a shorthand  
- Select the NDVI variable  
- Convert to a DataFrame using the `to_dataframe()` method  
- Join the two DataFrames for plotting using the `.join()` method. You will need to rename the columns using the `lsuffix =` and `rsuffix =` parameters
- Finally, plot annual July means for both inside and outside the Reservation on the same plot.

**Note**: The DateIndex in pandas is a little different from the Datetime Dimension in xarray. You will need to use the `.dt.year` syntax to access information about the year, not just `.year`.

In [6]:
### calculate mean annual July NDVI inside
july_ndvi_in_df = (
    ndvi_in_da

    ### group by year
    .groupby(ndvi_in_da.date.dt.year)

    ### calculate mean for each group
    .mean(...)

    ### extract NDVI and add to df
    .NDVI.to_dataframe()
)

### calculate mean annual July NDVI outside
july_ndvi_out_df = (
    ndvi_out_da

    ### group by year
    .groupby(ndvi_out_da.date.dt.year)

    ### calculate mean for each group
    .mean(...)

    ### extract NDVI and add to df
    .NDVI.to_dataframe()
)

In [7]:
### make df with in and out values
july_ndvi_df = (

    ### get NDVI in
    july_ndvi_in_df[['NDVI']]

    ### join with NDVI out
    .join(

        ### get NDVI out
        july_ndvi_out_df[['NDVI']],

        ### rename columns to be distinct
        lsuffix = ' inside GRIC', rsuffix = ' outside GRIC'
    )

)

july_ndvi_df

,NDVI inside GRIC,NDVI outside GRIC
year,,
2001,0.199645,0.247629
2002,0.177933,0.226726
2003,0.187301,0.229889
2004,0.176162,0.221753
2005,0.238630,0.255275
2006,0.211491,0.235571
2007,0.181710,0.211984
2008,0.201902,0.237739
2009,0.179118,0.218907


In [8]:
### plot inside and outside the reservation on a single plot
july_ndvi_inout_plot = july_ndvi_df.hvplot(
    title = "Mean July NDVI Inside vs Outside the GRIC Boundary",

)
display(july_ndvi_inout_plot)

:NdOverlay   [Variable]
   :Curve   [year]   (value)

Now, take the difference between outside and inside the site boundary and plot that


In [9]:
### calculate difference in mean July NDVI
july_ndvi_df['difference'] = (
    july_ndvi_df['NDVI inside GRIC'] 
    - july_ndvi_df['NDVI outside GRIC']
)


### plot difference inside and outside the boundary
july_ndvi_diff_plot = july_ndvi_df.difference.hvplot(
    title = "Difference in Mean July NDVI Inside vs Outside the GRIC Boundary",

)
display(july_ndvi_diff_plot)

:Curve   [year]   (difference)

In [10]:
hvplot.save(july_ndvi_diff_plot, 'july_ndvi_difference.html')

### Reflect and Respond #6:
What do you observe? Don’t forget to write a headline and description of your plot!

## Wrap up

Don’t forget to store your variables so you can use them in other notebooks! Replace `var1` and `var2` with the variable you want to save, separated by spaces.

In [11]:
%store july_ndvi_df july_ndvi_inout_plot july_ndvi_diff_plot

Stored 'july_ndvi_df' (DataFrame)
Stored 'july_ndvi_inout_plot' (NdOverlay)
Stored 'july_ndvi_diff_plot' (Curve)


Finally, be sure to `Restart` and `Run all` to make sure your notebook
works all the way through!